# Assignment 1

### Group Members

### Installation (Only Run if Needed)

In [ ]:
%pip install pymongo

### Import Packages

In [1]:
from pymongo import MongoClient, ReadPreference
from pymongo.write_concern import WriteConcern
from pymongo.read_concern import ReadConcern

import pymongo
import time

### Replica Set Connection

In [2]:
client = MongoClient(
    "mongodb://mongo1:27017,mongo2:27017,mongo3:27017/?replicaSet=dsa5208",
    serverSelectionTimeoutMS=5000,
    w=1,
    readConcernLevel="local"
)

In [3]:
hello = client.admin.command("hello")

print("Replica set:", hello.get("setName"))
print("Primary:", hello.get("primary"))
print("Writable primary:", hello.get("isWritablePrimary"))

Replica set: dsa5208
Primary: mongo1:27017
Writable primary: True


In [4]:
for name in ["mongo1", "mongo2", "mongo3"]:
    try:
        c = MongoClient(f"mongodb://{name}:27017/?directConnection=true", 
                        serverSelectionTimeoutMS=5000)
        info = c.admin.command("hello")
        print(name, "is connected to:", info.get("me"))
    except Exception as e:
        print(name, "FAILED:", e)

mongo1 is connected to: mongo1:27017
mongo2 is connected to: mongo2:27017
mongo3 is connected to: mongo3:27017


In [5]:
db = client.get_database('university', 
                         read_preference=ReadPreference.SECONDARY
                        )

#### Mongo 3

In [6]:
client2 = MongoClient(
    "mongodb://mongo2:27017/?directConnection=true",
    serverSelectionTimeoutMS=3000
)

client3 = MongoClient(
    "mongodb://mongo3:27017/?directConnection=true",
    serverSelectionTimeoutMS=3000
)

In [7]:
db3 = client3.get_database('university')

In [8]:
status = client.admin.command("replSetGetStatus")

for m in status["members"]:
    print(m["name"], m["stateStr"], m.get("optimeDate"))

mongo1:27017 PRIMARY 2026-09-09 15:48:37
mongo2:27017 SECONDARY 2026-09-09 15:48:27
mongo3:27017 SECONDARY 2026-09-09 15:48:27


#### Monotonic Reads

In [9]:
students = db["students"]

students.insert_one({
        "student_id": "S002",
        "student_name": "Feyd Johnston",
        "student_status": "Undergraduate",
        "senority" : "Year 1",
        "academic_load": "Full Time"
})

InsertOneResult(ObjectId('6aa1796c4f0412bf596af023'), acknowledged=True)

In [10]:
db.students.find_one({"student_id": "S002"})

{'_id': ObjectId('6aa1796c4f0412bf596af023'),
 'student_id': 'S002',
 'student_name': 'Feyd Johnston',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [11]:
db3.students.find_one({"student_id": "S002"})

{'_id': ObjectId('6aa1796c4f0412bf596af023'),
 'student_id': 'S002',
 'student_name': 'Feyd Johnston',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [12]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

{'count': 0,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788967276, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788967276, 1)}

In [19]:
status = client.admin.command("replSetGetStatus")

for m in status["members"]:
    print(m["name"], m["stateStr"], m.get("optimeDate"))

mongo1:27017 PRIMARY 2026-09-09 15:21:37
mongo2:27017 SECONDARY 2026-09-09 15:21:37
mongo3:27017 SECONDARY 2026-09-09 15:21:16


In [20]:
for i in range(2, 6):
    db.students.update_one(
        {"student_id": "S002"},
        {"$set": {"senority": f"Year {i}"}}
    )

In [21]:
no_session_reads = []
for _ in range(10):
    doc = db.students.with_options(read_preference=ReadPreference.SECONDARY).find_one(
        {"student_id": "S002"}
    )
    no_session_reads.append(doc["senority"])

print("No session:", no_session_reads)

No session: ['Year 5', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'Year 1', 'Year 1', 'Year 5', 'Year 1', 'Year 1']


In [22]:
db3.students.find_one({"student_id": "S002"})

{'_id': ObjectId('6aa1796c4f0412bf596af023'),
 'student_id': 'S002',
 'student_name': 'Feyd Johnston',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [25]:
with client.start_session(causal_consistency=True) as session:
    with_session_reads = []
    
    for _ in range(100):
        try:
            doc = db.students.with_options(read_preference=ReadPreference.SECONDARY).find_one(
                {"student_id": "S002"}, session=session, max_time_ms=3000
            )
            with_session_reads.append(doc["senority"])
        except Exception as e:
            with_session_reads.append("TIMEOUT")

print("With session:", with_session_reads)

With session: ['Year 1', 'Year 1', 'Year 1', 'Year 1', 'Year 5', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'TIMEOUT', 'Year 5', 'Year 5', 'TIMEOUT', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'TIMEOUT', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'TIMEOUT', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'TIMEOUT', 'Year 5', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'TIMEOUT', 'Year 5', 'Year 5', 'Year 5', 'TIMEOUT', 'Year 5', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'Year 5', 'Year 5', 'TIMEOUT', 'Year 5', 'Year 5', 'Year 5', 'TIMEOUT', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'Year 5', 'TIMEOUT', 'Year 5', 'TIMEOUT', 'TIMEOUT', '

In [26]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})

{'count': 2,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788967807, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788967276, 1)}

#### Monotonic Write

In [19]:
students = db["students"]

students.insert_one({
        "student_id": "S003",
        "student_name": "Alan White",
        "student_status": "Post Doc",
        "senority" : "Year 1",
        "academic_load": "Full Time"
})

InsertOneResult(ObjectId('6aa181c128fc72ff1bbeb5b2'), acknowledged=True)

In [20]:
db3.students.find_one({"student_id": "S003"})

{'_id': ObjectId('6aa181c128fc72ff1bbeb5b2'),
 'student_id': 'S003',
 'student_name': 'Alan White',
 'student_status': 'Post Doc',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [21]:
db.students.update_one(
    {"student_id": "S003"},
    {"$set": {"academic_history": ["B.Sc. Computer Science"]}},
    upsert=True
)

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000005'), 'opTime': {'ts': Timestamp(1788969415, 1), 't': 5}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1788969415, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1788969415, 1), 'updatedExisting': True}, acknowledged=True)

In [22]:
client2.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

{'count': 2,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788969415, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788969415, 1)}

In [24]:
time.sleep(3)
db.students.update_one({"student_id": "S003"}, {"$push": {"academic_history": "M.Sc. Data Science"}})
db.students.update_one({"student_id": "S003"}, {"$push": {"academic_history": "PhD Information Systems"}})

ServerSelectionTimeoutError: No primary available for writes, Timeout: 5.0s, Topology Description: <TopologyDescription id: 6aa17fcc28fc72ff1bbeb5ab, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('mongo1', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('mongo1:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>, <ServerDescription ('mongo2', 27017) server_type: RSSecondary, rtt: 0.001266455285793955>, <ServerDescription ('mongo3', 27017) server_type: RSSecondary, rtt: 0.0013655433200890498>]>

In [25]:
print("mongo3 mid-stall:", db3.students.find_one({"student_id": "S003"})["academic_history"])

mongo3 mid-stall: ['B.Sc. Computer Science']


In [26]:
db.students.find_one({"student_id": "S003"})["academic_history"]

['B.Sc. Computer Science']

#### Stop Mongo1

In [27]:
client2.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})

client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})

{'count': 4,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788969481, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788969415, 1)}

In [28]:
db.students.find_one({"student_id": "S003"})["academic_history"]

['B.Sc. Computer Science']

In [18]:
client.write_concern

WriteConcern(w=1)